In [1]:
import re
import math
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

print("Library berhasil diimpor.")

Library berhasil diimpor.


In [2]:
dokumen = {
    "D1": (
        "Sistem komputer membutuhkan perangkat keras dan perangkat lunak "
        "untuk menjalankan berbagai aplikasi."
    ),
    "D2": (
        "Jaringan komputer menghubungkan perangkat melalui jaringan lokal "
        "untuk berbagi data dan layanan."
    ),
    "D3": (
        "Kecerdasan buatan menggunakan algoritma dan data untuk membangun "
        "sistem yang mampu melakukan prediksi."
    ),
    "D4": (
        "Sistem temu kembali informasi membantu pengguna mencari dokumen "
        "berdasarkan data dan informasi yang relevan."
    )
}

print("Jumlah dokumen:", len(dokumen))
for doc_id, teks in dokumen.items():
    print(f"{doc_id}: {teks}")

Jumlah dokumen: 4
D1: Sistem komputer membutuhkan perangkat keras dan perangkat lunak untuk menjalankan berbagai aplikasi.
D2: Jaringan komputer menghubungkan perangkat melalui jaringan lokal untuk berbagi data dan layanan.
D3: Kecerdasan buatan menggunakan algoritma dan data untuk membangun sistem yang mampu melakukan prediksi.
D4: Sistem temu kembali informasi membantu pengguna mencari dokumen berdasarkan data dan informasi yang relevan.


In [3]:
stopwords = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan",
    "pada", "dalam", "atau", "adalah", "ini", "itu", "agar",
    "melalui", "berdasarkan", "mampu", "akan", "sebuah"
}

def preprocess_text(text):
    # Case folding
    text = text.lower()

    # Cleaning
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenisasi
    tokens = text.split()

    # Stopwords removal
    tokens = [token for token in tokens if token not in stopwords]

    return tokens

tokens_dokumen = {}

for doc_id, teks in dokumen.items():
    tokens_dokumen[doc_id] = preprocess_text(teks)

for doc_id, tokens in tokens_dokumen.items():
    print(f"{doc_id}: {tokens}")

D1: ['sistem', 'komputer', 'membutuhkan', 'perangkat', 'keras', 'perangkat', 'lunak', 'menjalankan', 'berbagai', 'aplikasi']
D2: ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'jaringan', 'lokal', 'berbagi', 'data', 'layanan']
D3: ['kecerdasan', 'buatan', 'menggunakan', 'algoritma', 'data', 'membangun', 'sistem', 'melakukan', 'prediksi']
D4: ['sistem', 'temu', 'kembali', 'informasi', 'membantu', 'pengguna', 'mencari', 'dokumen', 'data', 'informasi', 'relevan']


In [4]:
# Membuat vocabulary dari seluruh dokumen
vocabulary = sorted(set(
    token
    for tokens in tokens_dokumen.values()
    for token in tokens
))

print("Jumlah term:", len(vocabulary))
print("Vocabulary:")
print(vocabulary)

# Matriks Bag-of-Words berdasarkan raw count
bow = {}

for doc_id, tokens in tokens_dokumen.items():
    bow[doc_id] = {
        term: tokens.count(term)
        for term in vocabulary
    }

bow_df = pd.DataFrame(bow).T
bow_df

Jumlah term: 30
Vocabulary:
['algoritma', 'aplikasi', 'berbagai', 'berbagi', 'buatan', 'data', 'dokumen', 'informasi', 'jaringan', 'kecerdasan', 'kembali', 'keras', 'komputer', 'layanan', 'lokal', 'lunak', 'melakukan', 'membangun', 'membantu', 'membutuhkan', 'mencari', 'menggunakan', 'menghubungkan', 'menjalankan', 'pengguna', 'perangkat', 'prediksi', 'relevan', 'sistem', 'temu']


,algoritma,aplikasi,berbagai,berbagi,buatan,data,dokumen,informasi,jaringan,kecerdasan,...,mencari,menggunakan,menghubungkan,menjalankan,pengguna,perangkat,prediksi,relevan,sistem,temu
D1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,1,0,2,0,0,1,0
D2,0,0,0,1,0,1,0,0,2,0,...,0,0,1,0,0,1,0,0,0,0
D3,1,0,0,0,1,1,0,0,0,1,...,0,1,0,0,0,0,1,0,1,0
D4,0,0,0,0,0,1,1,2,0,0,...,1,0,0,0,1,0,0,1,1,1


In [5]:
tf = {}

for doc_id, tokens in tokens_dokumen.items():
    total_token = len(tokens)

    tf[doc_id] = {
        term: bow_df.loc[doc_id, term] / total_token
        for term in vocabulary
    }

tf_df = pd.DataFrame(tf).T

print("Jumlah token setiap dokumen:")
for doc_id, tokens in tokens_dokumen.items():
    print(f"{doc_id}: {len(tokens)} token")

print("\nMatriks TF:")
tf_df.round(4)

Jumlah token setiap dokumen:
D1: 10 token
D2: 9 token
D3: 9 token
D4: 11 token

Matriks TF:


,algoritma,aplikasi,berbagai,berbagi,buatan,data,dokumen,informasi,jaringan,kecerdasan,...,mencari,menggunakan,menghubungkan,menjalankan,pengguna,perangkat,prediksi,relevan,sistem,temu
D1,0.0000,0.1,0.1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.1,0.0000,0.2000,0.0000,0.0000,0.1000,0.0000
D2,0.0000,0.0,0.0,0.1111,0.0000,0.1111,0.0000,0.0000,0.2222,0.0000,...,0.0000,0.0000,0.1111,0.0,0.0000,0.1111,0.0000,0.0000,0.0000,0.0000
D3,0.1111,0.0,0.0,0.0000,0.1111,0.1111,0.0000,0.0000,0.0000,0.1111,...,0.0000,0.1111,0.0000,0.0,0.0000,0.0000,0.1111,0.0000,0.1111,0.0000
D4,0.0000,0.0,0.0,0.0000,0.0000,0.0909,0.0909,0.1818,0.0000,0.0000,...,0.0909,0.0000,0.0000,0.0,0.0909,0.0000,0.0000,0.0909,0.0909,0.0909


In [6]:
df_values = {}

for term in vocabulary:
    df_values[term] = sum(
        1 for tokens in tokens_dokumen.values()
        if term in tokens
    )

df_df = pd.DataFrame(
    {
        "Term": vocabulary,
        "DF": [df_values[term] for term in vocabulary]
    }
).set_index("Term")

df_df

,DF
Term,
algoritma,1
aplikasi,1
berbagai,1
berbagi,1
buatan,1
data,3
dokumen,1
informasi,1
jaringan,1


In [7]:
N = len(dokumen)

idf_values = {
    term: math.log(N / df_values[term])
    for term in vocabulary
}

idf_df = pd.DataFrame(
    {
        "Term": vocabulary,
        "DF": [df_values[term] for term in vocabulary],
        "IDF": [idf_values[term] for term in vocabulary]
    }
).set_index("Term")

idf_df.round(4)

,DF,IDF
Term,,
algoritma,1,1.3863
aplikasi,1,1.3863
berbagai,1,1.3863
berbagi,1,1.3863
buatan,1,1.3863
data,3,0.2877
dokumen,1,1.3863
informasi,1,1.3863
jaringan,1,1.3863


In [8]:
tfidf_manual = {}

for doc_id in dokumen:
    tfidf_manual[doc_id] = {
        term: tf[doc_id][term] * idf_values[term]
        for term in vocabulary
    }

tfidf_manual_df = pd.DataFrame(tfidf_manual).T

print("Matriks TF-IDF Manual:")
tfidf_manual_df.round(4)

Matriks TF-IDF Manual:


,algoritma,aplikasi,berbagai,berbagi,buatan,data,dokumen,informasi,jaringan,kecerdasan,...,mencari,menggunakan,menghubungkan,menjalankan,pengguna,perangkat,prediksi,relevan,sistem,temu
D1,0.000,0.1386,0.1386,0.000,0.000,0.0000,0.000,0.0000,0.0000,0.000,...,0.000,0.000,0.000,0.1386,0.000,0.1386,0.000,0.000,0.0288,0.000
D2,0.000,0.0000,0.0000,0.154,0.000,0.0320,0.000,0.0000,0.3081,0.000,...,0.000,0.000,0.154,0.0000,0.000,0.0770,0.000,0.000,0.0000,0.000
D3,0.154,0.0000,0.0000,0.000,0.154,0.0320,0.000,0.0000,0.0000,0.154,...,0.000,0.154,0.000,0.0000,0.000,0.0000,0.154,0.000,0.0320,0.000
D4,0.000,0.0000,0.0000,0.000,0.000,0.0262,0.126,0.2521,0.0000,0.000,...,0.126,0.000,0.000,0.0000,0.126,0.0000,0.000,0.126,0.0262,0.126


In [9]:
term_contoh = "jaringan"
doc_contoh = "D2"

jumlah_kemunculan = bow_df.loc[doc_contoh, term_contoh]
jumlah_token = len(tokens_dokumen[doc_contoh])
df_term = df_values[term_contoh]
idf_term = idf_values[term_contoh]
tf_term = tf[doc_contoh][term_contoh]
tfidf_term = tfidf_manual[doc_contoh][term_contoh]

print(f"Term: {term_contoh}")
print(f"Dokumen: {doc_contoh}")
print()
print(f"TF = {jumlah_kemunculan} / {jumlah_token} = {tf_term:.4f}")
print(f"DF = {df_term}")
print(f"IDF = log({N} / {df_term}) = {idf_term:.4f}")
print(f"TF-IDF = {tf_term:.4f} × {idf_term:.4f} = {tfidf_term:.4f}")

Term: jaringan
Dokumen: D2

TF = 2 / 9 = 0.2222
DF = 1
IDF = log(4 / 1) = 1.3863
TF-IDF = 0.2222 × 1.3863 = 0.3081


In [10]:
# Menggunakan teks hasil preprocessing
teks_preprocessed = {
    doc_id: " ".join(tokens)
    for doc_id, tokens in tokens_dokumen.items()
}

vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r'(?u)\b\w+\b'
)

tfidf_sklearn_matrix = vectorizer.fit_transform(
    teks_preprocessed.values()
)

tfidf_sklearn_df = pd.DataFrame(
    tfidf_sklearn_matrix.toarray(),
    index=dokumen.keys(),
    columns=vectorizer.get_feature_names_out()
)

print("Matriks TF-IDF dari scikit-learn:")
tfidf_sklearn_df.round(4)

Matriks TF-IDF dari scikit-learn:


,algoritma,aplikasi,berbagai,berbagi,buatan,data,dokumen,informasi,jaringan,kecerdasan,...,mencari,menggunakan,menghubungkan,menjalankan,pengguna,perangkat,prediksi,relevan,sistem,temu
D1,0.0000,0.3242,0.3242,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.3242,0.0000,0.5112,0.0000,0.0000,0.2069,0.0000
D2,0.0000,0.0000,0.0000,0.3219,0.0000,0.2055,0.0000,0.0000,0.6438,0.0000,...,0.0000,0.0000,0.3219,0.0000,0.0000,0.2538,0.0000,0.0000,0.0000,0.0000
D3,0.3577,0.0000,0.0000,0.0000,0.3577,0.2283,0.0000,0.0000,0.0000,0.3577,...,0.0000,0.3577,0.0000,0.0000,0.0000,0.0000,0.3577,0.0000,0.2283,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.0000,0.1857,0.2909,0.5819,0.0000,0.0000,...,0.2909,0.0000,0.0000,0.0000,0.2909,0.0000,0.0000,0.2909,0.1857,0.2909


In [11]:
print("Shape TF-IDF manual :", tfidf_manual_df.shape)
print("Shape TF-IDF sklearn :", tfidf_sklearn_df.shape)

print("\nTF-IDF Manual:")
print(tfidf_manual_df.round(4))

print("\nTF-IDF scikit-learn:")
print(tfidf_sklearn_df.round(4))

print("\nPerbedaan utama:")
print("- Manual: IDF = log(N / DF), tanpa smoothing dan tanpa normalisasi.")
print("- scikit-learn: IDF menggunakan smoothing dan hasil vektor dinormalisasi L2.")

Shape TF-IDF manual : (4, 30)
Shape TF-IDF sklearn : (4, 30)

TF-IDF Manual:
    algoritma  aplikasi  berbagai  berbagi  buatan    data  dokumen  \
D1      0.000    0.1386    0.1386    0.000   0.000  0.0000    0.000   
D2      0.000    0.0000    0.0000    0.154   0.000  0.0320    0.000   
D3      0.154    0.0000    0.0000    0.000   0.154  0.0320    0.000   
D4      0.000    0.0000    0.0000    0.000   0.000  0.0262    0.126   

    informasi  jaringan  kecerdasan  ...  mencari  menggunakan  menghubungkan  \
D1     0.0000    0.0000       0.000  ...    0.000        0.000          0.000   
D2     0.0000    0.3081       0.000  ...    0.000        0.000          0.154   
D3     0.0000    0.0000       0.154  ...    0.000        0.154          0.000   
D4     0.2521    0.0000       0.000  ...    0.126        0.000          0.000   

    menjalankan  pengguna  perangkat  prediksi  relevan  sistem   temu  
D1       0.1386     0.000     0.1386     0.000    0.000  0.0288  0.000  
D2       0.0000

In [12]:
for doc_id in dokumen:
    term_tertinggi = tfidf_manual_df.loc[doc_id].idxmax()
    nilai_tertinggi = tfidf_manual_df.loc[doc_id].max()

    print(
        f'{doc_id}: "{term_tertinggi}" '
        f'dengan bobot TF-IDF = {nilai_tertinggi:.4f}'
    )

D1: "aplikasi" dengan bobot TF-IDF = 0.1386
D2: "jaringan" dengan bobot TF-IDF = 0.3081
D3: "algoritma" dengan bobot TF-IDF = 0.1540
D4: "informasi" dengan bobot TF-IDF = 0.2521


ANALISIS

Berdasarkan perhitungan TF-IDF manual, term dengan bobot tertinggi pada setiap dokumen merupakan kta yang relatif lebih spesifik terhadap isi dokumen dibandingkan kata yang muncul pada banyak dokumen. Bobot TF-IDF menjadi tinggi ketika sebuah term cukup sering muncul dalam suatu dokumen tetapi jarang ditemukan pada dokumen lain. Karena itu, term seperti jaringan, kecerdasan, algoritma, atau temu kembali dapat
menjadi representasi penting dari topik dokumen masing-masing. Sebaliknya, term yang muncul pada
beberapa dokumen memiliki nilai IDF lebih rendah sehingga kontribusinya dalam membedakan dokumen juga lebih kecil. Representasi TF-IDF ini selanjutnya dapat digunakan sebagai dasar untuk menghitung kemiripan
antar dokumen menggunakan cosine similarity.